# ROI_processing

In [ ]:
import spatialdata_io
import spatialdata as sd
import troutpy
import scanpy as sc
from spatialdata import SpatialData
import matplotlib.pyplot as plt
from spatialdata.transformations import get_transformation
import matplotlib.patches as patches
import os
import numpy as np
import sys
import seaborn as sns
from matplotlib.colors import ListedColormap, Normalize
import matplotlib.pyplot as plt
import geopandas as gpd
from shapely.affinity import affine_transform
from shapely.geometry import box
import numpy as np
import pandas as pd
import hotspot
import gseapy

In [ ]:
# Technology 1: Xenium
sdata = sd.read_zarr("/media/sergio/Meninges/troutpy/rois/xenium_hsfemur/data_processed.zarr")

saving_path='/media/sergio/Meninges/troutpy/figures/hfemur'
os.makedirs(saving_path,exist_ok=True)

# Cell type annotation

In [ ]:
sc.tl.leiden(sdata['table'],resolution=1.2)
sc.pl.umap(sdata['table'],color='leiden')
sc.pl.spatial(sdata['table'],color='leiden',spot_size=100)
sc.tl.rank_genes_groups(sdata['table'],groupby='leiden')
sc.tl.dendrogram(sdata['table'],groupby='leiden')
sc.pl.rank_genes_groups_dotplot(sdata['table'],n_genes=4)
dict_anno={0:'Admixture 1',1:'Admixture 2',2:'Osteoblast',3:'Mesenchymal projenitor',4:'Osteoblast',5:'Admixture 4',6:'Endothelial',7:'Adipocyte',8:'Osteoblast',
9:'B cell',10:'Myeloid',11:'Osteoclast',12:'Pre-osteoblast',13:'NKCells_Tcells'} # 13 could be T cell also

In [ ]:
troutpy.pl.spatial_transcripts(sdata,gene_list=['FOXP2'],boundary_linewidth=0.04,use_roi=True,roi=(0,7000,3000,7000))

In [ ]:
troutpy.pl.spatial_transcripts(sdata,gene_list=['CEACAM8'],boundary_linewidth=0.04,use_roi=True,roi=(0,7000,3000,7000))

In [ ]:
sdata['table'].obs['cell_type']=sdata['table'].obs['leiden'].astype(int).map(dict_anno)

In [1]:
sdata['table'].uns['cell_type_colors'] = [
    "#0FCCAC",   # Aqua Teal – bright and fresh, great for highlighting
    "#ED2073",   # Vivid Magenta Pink – strong contrast, eye-catching
    "#ebb434", 
    "#BFCE00",   # Chartreuse Green – light, acidic green, good for uncommon categories
    "#DF5AFF",   # Electric Purple – vibrant, stands out, ideal for rare cell types
    "#FC911D",   # Vivid Orange – warm and bold, useful for attention-grabbing
    "#E891B4",   # Pastel Rose – soft and gentle, good for subtle categories
    "#eb3d34",   # Mint Cream – pale green, useful for background or low-contrast layers
    "#37BEE2",   # Bright Sky Blue – vibrant and energetic, visually accessible
    "#790DB7",   # Royal Purple – deep and rich, contrasts well with yellows and greens
    "#ff69f7",   # Lime Green – bold but not neon, works well for active states
    "#43FBFF",   # Neon Cyan – very bright, best with dark backgrounds
    "#F7C894",   # Peach – soft and neutral, ideal for subtle cell states
    "#BCBCBC",   # Medium Gray – neutral, useful as a baseline or control color
    "#5cc460",   # Charcoal – strong neutral, works well for text or background
    "#9CFFFC",
    "#123755",   # Deep Navy Blue – dark and cool, good background or strong category
    "#FFFF7B",   # Soft Lemon Yellow – light and readable on dark backgrounds
]


NameError: name 'sdata' is not defined

In [ ]:
import matplotlib.pyplot as plt
import matplotlib

# Set font type for vector output
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

sc.set_figure_params(scanpy=True, dpi=200, dpi_save=200, frameon=True, vector_friendly=True, fontsize=14, figsize=None, color_map=None, format='pdf', facecolor=None, transparent=False, ipython_format='retina')
sc.pl.umap(sdata['table'],color='cell_type',s=20,frameon=False,add_outline=True,show=False)
plt.savefig(os.path.join(saving_path,'cells_umap_celltype.pdf'))

sc.pl.spatial(sdata['table'],color='cell_type',spot_size=120,frameon=False, add_outline=True,show=False)
plt.savefig(os.path.join(saving_path,'cells_spatial_celltype.pdf'))

sc.set_figure_params(scanpy=True, dpi=100, dpi_save=150, frameon=True, vector_friendly=True, fontsize=14, figsize=None, color_map=None, format='pdf', facecolor=None, transparent=False, ipython_format='retina')

## Latent factor analysis

In [ ]:
troutpy.pp.aggregate_extracellular_transcripts(sdata,gene_key="gene",method='bin',square_size=20)

In [ ]:
adata = sdata.tables["segmentation_free_table"]
adata.raw = adata
sc.pp.filter_genes(adata, min_cells=2)
sc.pp.filter_cells(adata, min_genes=3)
sc.pp.filter_cells(adata, min_counts=10)

adata.layers["raw"] = adata.X.copy()
sc.pp.normalize_total(adata)
# Logarithmize the data
sc.pp.log1p(adata)
# pca
sc.tl.pca(adata)
# neighbors

In [ ]:
sc.pp.neighbors(adata, n_neighbors=30)
sc.tl.umap(adata)
sc.tl.leiden(adata, key_added="leiden", resolution=0.3)
sc.tl.rank_genes_groups(adata, "leiden", method="t-test", n_genes=10)
sc.pl.umap(adata, color="leiden", palette="tab20", s=30)
# plot 


In [ ]:
sc.pl.spatial(adata, color="leiden",spot_size=140)

## Plot exRNA profiles

In [ ]:
sc.tl.dendrogram(adata,groupby='leiden')
sc.pl.rank_genes_groups_dotplot(adata, groupby="leiden", n_genes=8,show=False)

In [ ]:
#dict_urna={0:'CA1-2sr-so',1:'CC',2:'CTX-DGpo',3:'Thalamus 1',4:'CAslm',5:'DGmo inner',6:'Astrocytic Gfap-',7:'CA3sr-so',8:'Thalamus 2',9:'Astrocytic Gfap+',10:'DGmo outer',11:'Low counts'}
adata.obs['urna_cluster']=adata.obs['leiden'].astype(int)#.map(dict_urna)
#del adata.uns['urna_cluster_colors']
adata.uns['urna_cluster_colors']=['#81fcc1','#99d1c3','#fc77aa','#ded67e','#8c66b0','#7fc777','#fa6e6b','#fcae77','#fcd2a4','#b5b3b5','#a4bafc','#637cc7']

### Plot the resulting map

In [ ]:
import matplotlib.pyplot as plt
import matplotlib

# Set font type for vector output
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

sc.set_figure_params(scanpy=True, dpi=200, dpi_save=200, frameon=True, vector_friendly=True, fontsize=14, figsize=None, color_map=None, format='pdf', facecolor=None, transparent=False, ipython_format='retina')
sc.pl.umap(adata, color="urna_cluster", s=6, show=False,alpha=1.0,frameon=False)
plt.savefig(os.path.join(saving_path,'umap_urna_clusters.pdf'))

import matplotlib.pyplot as plt
import matplotlib

# Set font type for vector output
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

sc.set_figure_params(scanpy=True, dpi=200, dpi_save=200, frameon=True, vector_friendly=True, fontsize=14, figsize=None, color_map=None, format='pdf', facecolor=None, transparent=False, ipython_format='retina')
sc.pl.spatial(adata, color="urna_cluster", spot_size=12, show=False,alpha=1.0,frameon=False)
plt.savefig(os.path.join(saving_path,'spatial_urna_clusters.pdf'))

sc.set_figure_params(scanpy=True, dpi=100, dpi_save=100, frameon=True, vector_friendly=True, fontsize=14, figsize=None, color_map=None, format='pdf', facecolor=None, transparent=False, ipython_format='retina')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
sc.pl.spatial(adata, color="urna_cluster", spot_size=12, show=False, ax=ax,marker='o',alpha=0.8)
#fig,ax=plt.subplots(figsize=(7,7))
#sc.pl.spatial(sdata["table"],color="cell_type",spot_size=40, ax=ax,add_outline=True)
sc.pl.spatial(sdata['table'],color='cell_type',spot_size=15,frameon=False, ax=ax,show=False,add_outline=True, outline_color=('black', 'black'))
plt.savefig(os.path.join(saving_path,'spatial_urna_clusters_and_celltypes.pdf'))

In [ ]:
sc.tl.rank_genes_groups(adata, groupby="urna_cluster")
sc.tl.dendrogram(adata,groupby='urna_cluster')

import matplotlib.pyplot as plt
import matplotlib

# Set font type for vector output
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

sc.set_figure_params(scanpy=True, dpi=200, dpi_save=200, frameon=True, vector_friendly=True, fontsize=14, figsize=None, color_map=None, format='pdf', facecolor=None, transparent=False, ipython_format='retina')
sc.pl.rank_genes_groups_dotplot(adata, groupby="urna_cluster", n_genes=4,cmap='Blues',show=False)
plt.savefig(os.path.join(saving_path,'urna_clusters_deg.pdf'))

# Mean signature similarity

In [ ]:
sdata['table'] ### cell type in 'cell_type'
adata ## cluster in 'urna_cluster'

### filter for common genes, get average gene expression profile for each
### Get cosine similarity between the clusters of each group

##represent as heatmap

In [ ]:
subset.X.mean(axis=0)[0,:][0,:]

In [ ]:
# 1. Filter for common genes
common_genes = list(set(sdata['table'].var_names) & set(adata.var_names))
import pandas as pd
import numpy as np

# Assuming common_genes is already defined and filtered
sdata_filtered = sdata['table'][:, common_genes]
adata_filtered = adata[:, common_genes]

# For sdata
cell_types = sdata_filtered.obs['cell_type'].unique()
cell_types=list(cell_types.astype(str))
cell_types.sort()
sdata_avg_expr = []
for cell_type in cell_types:
    subset = sdata_filtered[sdata_filtered.obs['cell_type'] == cell_type]
    mean_expr = np.asarray(subset.X.mean(axis=0)).flatten()  # convert to 1D array
    sdata_avg_expr.append(mean_expr)

sdata_avg_expr = pd.DataFrame(sdata_avg_expr, index=cell_types, columns=common_genes)

# For adata
clusters = adata_filtered.obs['urna_cluster'].unique()
clusters=list(clusters.astype(str))
clusters.sort()
adata_avg_expr = []
for cluster in clusters:
    subset = adata_filtered[adata_filtered.obs['urna_cluster'] == cluster]
    mean_expr = np.asarray(subset.X.mean(axis=0)).flatten()
    adata_avg_expr.append(mean_expr)

adata_avg_expr = pd.DataFrame(adata_avg_expr, index=clusters, columns=common_genes)

# Compute cosine similarity
from sklearn.metrics.pairwise import cosine_similarity
similarity = cosine_similarity(sdata_avg_expr.values, adata_avg_expr.values)

# Plot heatmap

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Set font type for vector output
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

sc.set_figure_params(scanpy=True, dpi=200, dpi_save=200, frameon=True, vector_friendly=True, fontsize=14, figsize=None, color_map=None, format='pdf', facecolor=None, transparent=False, ipython_format='retina')
plt.figure(figsize=(7, 7))
sns.heatmap(similarity,
            xticklabels=adata_avg_expr.index,
            yticklabels=sdata_avg_expr.index,
            cmap='Blues', annot=False, fmt=".1f")
plt.xlabel('urna_cluster')
plt.ylabel('cell_type')
plt.title('Cosine similarity of mean expression profiles')
plt.tight_layout()
plt.grid(False)
plt.savefig(os.path.join(saving_path,'cosine_similarity_urna_cell_types.pdf'))
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 8))
sns.heatmap(pearson_correlation_matrix,
            xticklabels=adata_avg_expr.index,
            yticklabels=sdata_avg_expr.index,
            cmap='viridis', annot=False, fmt=".1f")
plt.xlabel('urna_cluster')
plt.ylabel('cell_type')
plt.title('Cosine similarity of mean expression profiles')
plt.tight_layout()
plt.grid(False)
plt.show()

# Work on factor identification

In [ ]:
troutpy.pp.filter_xrna(sdata,min_logfoldratio_over_noise=0,gene_key='gene')
troutpy.pp.filter_xrna(sdata,gene_key='gene',control_probe=True)
troutpy.pp.filter_xrna(sdata,gene_key='gene',min_extracellular_proportion=0.1)

In [ ]:
troutpy.tl.latent_factor(sdata, method="NMF", layer="segmentation_free_table", n_components=10)
troutpy.tl.apply_exrna_factors_to_cells(sdata, extracellular_layer="segmentation_free_table", cellular_layer="table")


In [ ]:
from typing import Any

import anndata
import matplotlib.pyplot as plt
import numpy as np
import scanpy as sc
import seaborn as sns
from anndata import AnnData
from matplotlib import gridspec, rcParams
from spatialdata import SpatialData  # Assuming SpatialData is from scverse
from troutpy.pl.colors import get_colormap




In [ ]:
rank_factor_genes_loadings_matrixplot(sdata, layer="segmentation_free_table", n_genes=3,figsize=(4,7),cmap='Pinks_contrast',save=os.path.join(saving_path,'gene_loadings_factors.pdf'))

In [ ]:
sc.set_figure_params(scanpy=True, dpi=100, dpi_save=100, frameon=True, vector_friendly=True, fontsize=14, figsize=None, color_map=None, format='pdf', facecolor=None, transparent=False, ipython_format='retina')
troutpy.pl.rank_factor_genes_loadings_matrixplot(sdata, layer="segmentation_free_table", n_genes=1)
troutpy.pl.rank_factor_genes_loadings(sdata, layer="segmentation_free_table", n_genes=5)
sc.pl.umap(sdata['table'],color='cell_type')

In [ ]:
sdata['table'].obsm['factors_cell_loadings'].columns=sdata['table'].obsm['factors_cell_loadings'].columns.astype(str)

In [ ]:
#sdata.write("/media/sergio/Meninges/troutpy/rois/xenium_msbrain_5k/data_processed_factor.zarr",overwrite=True)

### Contiue with gene set enrichment analysis

In [ ]:
# %matplotlib inline
# %config InlineBackend.figure_format='retina' # mac
%load_ext autoreload
%autoreload 2
import pandas as pd
import gseapy as gp
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
uadata=sdata['segmentation_free_table']

In [ ]:
gene_loadings

In [ ]:

records = []

# Collect enrichment data for all factors, preserving order
for factor in range(10):
    #top_genes = list(gene_loadings.sort_values(by=factor, ascending=False).head(50).index)
    gnl=gene_loadings.sort_values(by=factor,ascending=False).copy()
    gn=gnl.div(gnl.sum(axis=0),axis=1)
    proportion=0.01 # this the proportion of contribution to the loading
    top_genes=list(gn[gn[factor]>proportion].index)
    
    enr = gp.enrichr(
        gene_list=top_genes,
        gene_sets=['GO_Cellular_Component_2021'],
        organism='Mouse',
        outdir=None,
        cutoff=0.5
    )
    df = enr.results
    if df.empty:
        continue
    
    # Filter by significance threshold, e.g., adjusted p-value < 0.05
    sig_df = df[df['Adjusted P-value'] < 0.01].sort_values('Adjusted P-value').reset_index(drop=True)
    #df = df.sort_values('Adjusted P-value').head(5).reset_index(drop=True)
    
    # Only keep significant terms, no fixed number limit
    for order, row in sig_df.iterrows():
        records.append({
            'Factor': factor,
            'Term': row['Term'],
            'AdjP': row['Adjusted P-value'],
            'NegLogAdjP': -np.log10(row['Adjusted P-value']),
            'GeneCount': int(row['Overlap'].split('/')[0]),  # extract numerator of overlap "x/y"
            'Zscore': row['Combined Score'],
            'Order': order  # preserve order within factor
        })

In [ ]:
simple_term=[term.split('(')[0] for term in enrich_df['Term']]
enrich_df['simple_term']=simple_term

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib
from matplotlib.colors import Normalize

# Set font type for vector output
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

# Define vmax for Zscore coloring, e.g., cap at 10
vmax_value = 2000

norm = Normalize(vmin=enrich_df['Zscore'].min(), vmax=vmax_value)

plt.figure(figsize=(3.5, 0.35 * len(term_order)))
scatter = sns.scatterplot(
    data=enrich_df,
    x='Factor',
    y='simple_term',
    size='NegLogAdjP',
    hue='Zscore',
    palette='YlGnBu',
    sizes=(70, 300),
    edgecolor='black',
    hue_norm=norm  # <-- this applies the normalization (vmax)
)

plt.xlabel('Factor')
plt.ylabel('GO Cellular Component Term')
plt.title('Enrichment Dotplot: Size=GeneCount, Color=Zscore (Combined Score)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(False)
plt.tight_layout()

plt.savefig(os.path.join(saving_path, 'gsea_compartment_brain.pdf'))
plt.show()


## Plot factors by cell type

In [ ]:
troutpy.pl.factors_in_cells(sdata, layer="table", cmap='Aquas_contrast', method="matrixplot",celltype_key='cell_type',standard_scale='var',show=False)
plt.savefig(os.path.join(saving_path, 'factors_scores_per_celltype_brain.pdf'))

In [ ]:
troutpy.pl.factors_in_cells(sdata, layer="table", cmap='YlGnBu', method="matrixplot",celltype_key='cell_type',standard_scale='var',show=False)
plt.savefig(os.path.join(saving_path, 'factors_scores_per_celltype_brain_YlGnBu.pdf'))

In [ ]:
troutpy.pl.factors_in_cells(sdata, layer="table",cmap='BuPi', method="heatmap",celltype_key='cell_type',standard_scale='var',show=False)
plt.savefig(os.path.join(saving_path, 'factors_scores_per_celltype_brain_heatmap.pdf'))

In [ ]:
sdata['segmentation_free_table'].varm['gene_loadings']

In [ ]:
# This is not very nicely built
troutpy.pl.rank_factor_genes_loadings_matrixplot(sdata, layer="segmentation_free_table", n_genes=5,cmap='viridis',show=False)

# Plot factor loadings in space

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import scanpy as sc
import troutpy as tp

def plot_factor_loadings(sdata, layer_cells="table", layer_extra="segmentation_free_table", spot_size_cells=7, spot_size_extra=12,complete_saving_path=None):
    """
    Plots factor loadings for each factor in a single figure with subplots,showing both intra- and extra-cellular scores.

    Parameters
    ----------
    sdata
        SpatialData object containing cell and extra-cellular data.

    layer_cells
        Key in sdata where *cell* factor loadings are stored.

    layer_extra
        Key in sdata where *extra-cellular* factor loadings are stored.

    spot_size_cells
        Size of the marker for cells.

    spot_size_extra
        Size of the marker for extra-cellular spots.
    """
    cell_loadings = sc.AnnData(sdata[layer_cells].obsm["factors_cell_loadings"], obs=sdata[layer_cells].obs)
    cell_loadings.var.index = [f"Factor {i + 1}" for i in range(cell_loadings.shape[1])]
    cell_loadings.obsm["spatial"] = sdata[layer_cells].obsm["spatial"]

    extra_loadings = sc.AnnData(sdata[layer_extra].obsm["cell_loadings"], obs=sdata[layer_extra].obs)
    extra_loadings.var.index = [f"Factor {i + 1}" for i in range(extra_loadings.shape[1])]
    extra_loadings.obsm["spatial"] = sdata[layer_extra].obsm["spatial"]

    n_factors = cell_loadings.shape[1]

    cold_palette = tp.pl.get_colormap("Aquas_contrast")  # For intra-cellular factors
    warm_palette = tp.pl.get_colormap("Pinks")  # For extra-cellular factors

    n_cols = 2  # Number of columns in the grid
    n_rows = int(np.ceil(n_factors / n_cols))  # Compute number of rows needed
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(10 * n_cols, 10 * n_rows))
    axes = axes.flatten()  # Flatten in case of single row

    for i in range(n_factors):
        ax = axes[i]
        factor_name = f"Factor {i + 1}"

        # Plot extra-cellular loadings (background)
        sc.pl.spatial(
            extra_loadings, color=factor_name, spot_size=spot_size_extra, show=False, ax=ax, cmap=warm_palette, marker="*", colorbar_loc=None
        )

        # Plot intra-cellular loadings (foreground with black outline)
        sc.pl.spatial(
            cell_loadings,
            color=factor_name,
            spot_size=spot_size_cells,
            show=False,
            ax=ax,
            cmap=cold_palette,
            edgecolor="black",
            linewidth=0.03,
            colorbar_loc=None, vmax='p99'
        )

        ax.set_title(factor_name)

    # Remove empty subplots if n_factors is not a multiple of n_cols
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])

    plt.tight_layout()
    if complete_saving_path is not None:
        plt.savefig(complete_saving_path)
    plt.show()

In [ ]:
plot_factor_loadings(sdata, layer_cells="table", layer_extra="segmentation_free_table", spot_size_cells=50,spot_size_extra=50,complete_saving_path=os.path.join(saving_path,'factors_urna_cells.pdf'))

In [ ]:
factors_adata=sc.AnnData(sdata['segmentation_free_table'].obsm['cell_loadings'],obs=sdata['segmentation_free_table'].obs)

In [ ]:
sc.pl.heatmap(factors_adata,factors_adata.var.index,groupby='urna_cluster',cmap=troutpy.pl.get_colormap('Aquas'),show=False)
plt.savefig(os.path.join(saving_path, 'factors_x_urna_cluster_heatmap.pdf'))
sc.pl.dotplot(factors_adata,factors_adata.var.index,groupby='urna_cluster',cmap=troutpy.pl.get_colormap('Aquas'),show=False)
plt.savefig(os.path.join(saving_path, 'factors_x_urna_cluster_dotplot.pdf'))

## Multimodal analysis (protein addition)

In [ ]:
troutpy.tl.image_intensities_per_transcript(sdata, image_key="morphology_focus", scale="scale0", transcript_key="transcripts", extracellular=True, copy=False)

In [ ]:
ad = sdata["image_intensity_per_transcript"]

In [ ]:
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
import anndata as ad

# Parameters
k = 10  # number of nearest neighbors

# Step 1: Extract spatial coordinates
coords_segfree = sdata['segmentation_free_table'].obsm['spatial']
coords_intensity = sdata['image_intensity_per_transcript'].obsm['spatial']

# Step 2: Build KDTree from image_intensity
tree = cKDTree(coords_intensity)

# Step 3: Query k nearest neighbors for each point in segmentation_free_table
dists, idxs = tree.query(coords_segfree, k=k)

# Step 4: Get expression matrix from image_intensity
img_expr = sdata['image_intensity_per_transcript'].to_df().values  # shape: (n_cells_img, n_genes)

# Step 5: Average expression across k neighbors for each query point
avg_expr = np.array([img_expr[inds].mean(axis=0) for inds in idxs])  # shape: (n_cells_segfree, n_genes)

# Step 6: Create a new AnnData object to store approximated values
approximated_adata = ad.AnnData(X=avg_expr)
approximated_adata.var_names = sdata['image_intensity_per_transcript'].var_names
approximated_adata.obs_names = sdata['segmentation_free_table'].obs_names
approximated_adata.obsm['spatial'] = coords_segfree  # reuse segfree spatial positions

factor_loadings=pd.DataFrame(sdata['segmentation_free_table'].obsm['cell_loadings'])


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

cor_matrix = pd.DataFrame(
    np.corrcoef(factor_loadings.T, approximated_adata.X.T)[
        :factor_loadings.shape[1], factor_loadings.shape[1]:  # select factor-gene block
    ],
    index=factor_loadings.columns,
    columns=approximated_adata.var_names,
)

cor_matrix=cor_matrix.fillna(0)
cor_matrix=cor_matrix.loc[:,~cor_matrix.columns.isin(['dummy'])]
# Plot a heatmap of top-correlated genes per factor
top_genes = cor_matrix.abs().max().nlargest(10).index
sns.clustermap(cor_matrix[top_genes].transpose(), cmap=troutpy.pl.get_colormap('BuPi'), center=0, figsize=(8, 2),vmax=0.5,vmin=-0.5,row_cluster=False, col_cluster=False)
plt.suptitle("PCC Factors and Staining intensities",fontsize=10)
plt.savefig(os.path.join(saving_path, 'multimodal_quantification.pdf'))
plt.show()


In [ ]:
approximated_adata.var.index.isin(['dummy'])

In [ ]:
factor_loadings

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os
from sklearn.preprocessing import scale

# Set threshold for selecting positive cells
threshold = 0.1  # Adjust if needed

approximated_adata=approximated_adata[:,~approximated_adata.var.index.isin(['dummy'])]
# Compute global mean protein expression (across all cells)
global_means = np.asarray(approximated_adata.X.mean(axis=0)).ravel()

# Compute per-factor mean expression for positive cells
mean_exprs = []
valid_factors = []

for factor in factor_loadings.columns:
    positive_cells = factor_loadings[factor] > threshold
    if positive_cells.sum() == 0:
        continue
    mean_expr = np.asarray(approximated_adata.X[positive_cells, :]).mean(axis=0)
    mean_exprs.append(mean_expr)
    valid_factors.append(factor)

# Build DataFrame of mean expressions
mean_exprs_df = pd.DataFrame(mean_exprs, index=valid_factors, columns=approximated_adata.var_names)

# Remove 'dummy' markers if present
mean_exprs_df = mean_exprs_df.loc[:, ~mean_exprs_df.columns.isin(['dummy'])]
global_means = pd.Series(global_means, index=approximated_adata.var_names)
global_means = global_means[mean_exprs_df.columns]  # align columns

# ==== Z-score per marker ====
zscored_df = (mean_exprs_df - global_means) / np.std(approximated_adata.X, axis=0)

# Plot heatmap (choose either DataFrame)
plt.figure(figsize=(10, 4))
sns.heatmap(
    zscored_df.transpose(),  # or mean_centered_df
    cmap=troutpy.pl.get_colormap('BuPi'),
    center=0,
    cbar_kws={"label": "Z-scored Protein Expression"}, vmax=1, vmin=-1
)
plt.ylabel("Protein Marker")
plt.xlabel("Factor")
plt.title("Z-scored Protein Expression in Positive Cells per Factor")
plt.tight_layout()
plt.grid(False)
plt.savefig(os.path.join(saving_path, 'protein_expression_contextualized.pdf'))
plt.show()


In [ ]:
sdata['image_intensity_per_transcript']
sdata['segmentation_free_table']

In [ ]:
ad=approximated_adata
sc.pl.spatial(ad, color=ad.var.index, spot_size=11, cmap=troutpy.pl.get_colormap('Aquas_contrast'), vmax="p99", ncols=2,show=False)
plt.savefig(os.path.join(saving_path, 'spatial_map_proteins_aquas.pdf'))
ad=approximated_adata
sc.pl.spatial(ad, color=ad.var.index, spot_size=11, cmap=troutpy.pl.get_colormap('BuPi'), vmax="p99", ncols=2,show=False)
plt.savefig(os.path.join(saving_path, 'spatial_map_proteins_BuPi.pdf'))

In [ ]:
exp=ad.to_df()
exp['gene']=ad.obs['gene']

In [ ]:
mean_intensity=exp.groupby('gene').mean()

In [ ]:
mean_intensity.sort_values(by='ATP1A1/CD45/E-Cadherin')

# Module analysis

In [ ]:
adata = sdata["table"]
hs = hotspot.Hotspot(adata, layer_key="raw", model="danb", latent_obsm_key="X_pca", umi_counts_obs_key="n_counts")

hs.create_knn_graph(weighted_graph=False, n_neighbors=10)
hs_results = hs.compute_autocorrelations()
hs_genes = hs_results.loc[hs_results.FDR < 0.005].index  # Select genes
local_correlations = hs.compute_local_correlations(hs_genes, jobs=4)  # jobs for parallelization

In [ ]:
# calculate modules
modules = hs.create_modules(min_gene_threshold=10, core_only=True, fdr_threshold=0.0005)
module_scores = hs.calculate_module_scores()

In [ ]:
hs.plot_local_correlations(vmax=10, vmin=-10)

In [ ]:
adata_scores = sc.AnnData(module_scores, obs=adata.obs)
adata_scores.obsm["X_umap"] = adata.obsm["X_umap"]
adata_scores.obsm["X_pca"] = adata.obsm["X_pca"]
sc.pl.umap(adata, color="leiden")
sc.pl.umap(adata_scores, color=adata_scores.var.index, vmax="p99")

In [ ]:
dictmod = dict(zip(modules.index, modules.values))
results=sdata['xrna_metadata'].var
results["module"] = results.index.map(dictmod)

In [ ]:
import matplotlib.pyplot as plt


def annotate_top_bottom(
    df, metric1, metric2, n=5, label_col=None, figsize=(8, 6), scatter_kwargs=None, annotate_kwargs=None, title_tag=""
):
    """
    Scatter-plot df[metric1] vs df[metric2] and annotate the top/bottom n points
    along each axis.

    Parameters
    ----------
    df : pandas.DataFrame
        The data.
    metric1 : str
        Column name for the x-axis metric.
    metric2 : str
        Column name for the y-axis metric.
    n : int, optional (default=5)
        Number of top and bottom points to label per metric.
    label_col : str or None, optional (default=None)
        Column name containing the labels. If None, uses df.index.
    figsize : tuple, optional (default=(8, 6))
        Figure size.
    scatter_kwargs : dict or None
        Extra kwargs passed to plt.scatter().
    annotate_kwargs : dict or None
        Extra kwargs passed to plt.annotate().
    Title tag: str, optional (default='')

    Returns
    -------
    fig, ax : matplotlib.figure.Figure, matplotlib.axes.Axes
        The figure and axes objects.
    """
    # Prepare labels
    df_plot = df.copy()
    if label_col is None:
        df_plot["__idx_label"] = df_plot.index.astype(str)
        label_col = "__idx_label"

    # Plot
    fig, ax = plt.subplots(figsize=figsize)
    sc_kwargs = {} if scatter_kwargs is None else scatter_kwargs
    ax.scatter(df_plot[metric1], df_plot[metric2], **sc_kwargs)
    ax.set_xlabel(metric1)
    ax.set_ylabel(metric2)
    ax.set_title(f"{title_tag}: {metric1} vs {metric2}")

    # Find indices to annotate
    top1 = df_plot.nlargest(n, metric1).index
    bot1 = df_plot.nsmallest(n, metric1).index
    top2 = df_plot.nlargest(n, metric2).index
    bot2 = df_plot.nsmallest(n, metric2).index
    to_annotate = set(top1) | set(bot1) | set(top2) | set(bot2)

    # Annotate
    ann_kwargs = dict(xytext=(5, 5), textcoords="offset points", ha="left", va="bottom", fontsize=8, alpha=0.8)
    if annotate_kwargs:
        ann_kwargs.update(annotate_kwargs)

    for idx in to_annotate:
        x = df_plot.loc[idx, metric1]
        y = df_plot.loc[idx, metric2]
        label = df_plot.loc[idx, label_col]
        ax.annotate(label, xy=(x, y), **ann_kwargs)

    fig.tight_layout()
    return fig, ax

In [ ]:
results2 = results[results["logfoldratio_over_noise"] > 0]
fig, ax = annotate_top_bottom(
    df=results2,
    metric1="logfoldratio_over_noise",
    metric2="in_out_spearmanR",
    n=10,
    scatter_kwargs={"alpha": 0.6, "s": 30},
    annotate_kwargs={"fontsize": 9, "color": "black"},
    figsize=(7, 7),
)
plt.show()

In [ ]:
fig, ax = annotate_top_bottom(
    df=results2,
    metric1="logfoldratio_over_noise",
    metric2="extracellular_proportion",
    n=50,
    scatter_kwargs={"alpha": 0.6, "s": 30},
    annotate_kwargs={"fontsize": 9, "color": "black"},
    figsize=(7, 7),
)
plt.show()

In [ ]:
# spatial maps-in out
g2 = "Gap43"
troutpy.pl.spatial_inout_expression(sdata, g2, spot_size_cells=10)

In [ ]:
for g, i in results.groupby("module"):
    if g > -1:
        print(g)
        # Suppose `results` is your DataFrame and you have a column 'gene' with names:
        fig, ax = annotate_top_bottom(
            df=i,
            metric1="logfoldratio_over_noise",
            metric2="extracellular_proportion",
            n=5,
            scatter_kwargs={"alpha": 0.6, "s": 30},
            annotate_kwargs={"fontsize": 9, "color": "black"},
            title_tag=g,
            figsize=(4, 4),
        )
        plt.show()

In [ ]:
for g, i in results.groupby("module"):
    if g > -1:
        print(g)
        # Suppose `results` is your DataFrame and you have a column 'gene' with names:
        fig, ax = annotate_top_bottom(
            df=i,
            metric1="logfoldratio_over_noise",
            metric2="in_out_spearmanR",
            n=5,
            scatter_kwargs={"alpha": 0.6, "s": 30},
            annotate_kwargs={"fontsize": 9, "color": "black"},
            title_tag=g,
            figsize=(4, 4),
        )
        plt.show()

In [ ]:
import pandas as pd

mods_df = pd.DataFrame(modules).sort_values(by="Module")
genes = mods_df[mods_df["Module"] == 1].index
sc.pl.umap(adata, color=genes, vmax="p99")
print(genes)

In [ ]:
g2 = "CAVIN2"
troutpy.pl.spatial_inout_expression(sdata, g2, spot_size_cells=40, spot_size_extra=40)

In [ ]:
g2 = "FABP4"
troutpy.pl.spatial_inout_expression(sdata, g2, spot_size_cells=40, spot_size_extra=40)

In [ ]:
merged_info.sort_values(by=[metric + "_cosmx"], ascending=False)

In [ ]:
fig, ax = annotate_top_bottom(
    df=results,
    metric1="logfoldratio_over_noise",
    metric2="extracellular_proportion",
    n=5,
    scatter_kwargs={"alpha": 0.6, "s": 30},
    annotate_kwargs={"fontsize": 9, "color": "black"},
    title_tag=g,
    figsize=(4, 4),
)
plt.show()

In [ ]:
g2 = "FZD4"
troutpy.pl.spatial_inout_expression(sdata, g2, spot_size_cells=10, spot_size_extra=10)

In [ ]:
trans = sdata["transcripts"].compute()

In [ ]:
trans["segstatus"] = trans["overlaps_cell"].astype(str) + "_" + trans["extracellular"].astype(str)
stat2lab = {"True_False": "cell", "False_True": "extracellular", "False_False": "cell-like"}

In [ ]:
trans["type"] = trans["segstatus"].map(stat2lab)

In [ ]:
trans2 = trans.loc[:, ["x", "y", "gene", "type"]]

In [ ]:
trans2.to_csv("/media/sergio/Meninges/trans2.csv")